In [12]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl
from collections import Counter
import torch

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import classification_report, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler


# Importación de los modelos destacados
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings('ignore')
print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [13]:

# 1. Configurar dispositivo y variables
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

base_dir = r"C:\Users\Usuario\Documents\Workspace\Mirage\TFG\dataset\dataset_finales\VAE"
batch_size = 32  # Asegurar definición de batch_size

dataset_type = "codigos"

# 2. Rutas del dataset VAE
if dataset_type == "codigos":
    train_path = os.path.join(base_dir, "codigos_train_VAE.csv")
    test_path = os.path.join(base_dir, "codigos_test_VAE.csv")
elif dataset_type == "descripciones":
    train_path = os.path.join(base_dir, "descripciones_train_VAE.csv")
    test_path = os.path.join(base_dir, "descripciones_test_VAE.csv")
else:  # combinado
    train_path = os.path.join(base_dir, "combinado_train_VAE.csv")
    test_path = os.path.join(base_dir, "combinado_test_VAE.csv")

df_train = pd.read_csv(train_path, sep="|")
df_test = pd.read_csv(test_path, sep="|")

# 3. Filtrar por códigos objetivo
codes = sorted(["F20", "F21", "F22", "F23", "F25", "F29", "F60.1"])
df_train = df_train[df_train["DIAG PSQ"].isin(codes)].copy()
df_test = df_test[df_test["DIAG PSQ"].isin(codes)].copy()

# 4. Filtrado estricto de características numéricas del VAE
target_col = "DIAG PSQ"
columnas_basura = [target_col, "id", "text", "Unnamed: 0", "label", "Origen"]

# Excluir basura y columnas que empiecen por 'Diag'/'diag'
feature_cols = [
    col for col in df_train.columns 
    if col not in columnas_basura and not col.lower().startswith("diag")
]

# Seleccionar SOLO columnas numéricas y reemplazar NaNs por 0
X_train_df = df_train[feature_cols].select_dtypes(include=[np.number]).fillna(0)
X_test_df = df_test[feature_cols].select_dtypes(include=[np.number]).fillna(0)

X_train = X_train_df.values
X_test = X_test_df.values

# 5. Codificar etiquetas alineadas
cat_type = pd.CategoricalDtype(categories=codes, ordered=True)
y_train = df_train[target_col].astype(cat_type).cat.codes.values
y_test = df_test[target_col].astype(cat_type).cat.codes.values

num_classes = len(codes)
input_dim = X_train.shape[1]

# 6. Escalado de características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# -------------------------------------------------------------------
# OPCIÓN A: Para modelos Scikit-Learn, LightGBM y XGBoost
# Usar: X_train_scaled, y_train, X_test_scaled, y_test
# -------------------------------------------------------------------

# -------------------------------------------------------------------
# OPCIÓN B: Para Redes Neuronales / PyTorch / BERT Embeddings VAE
# Usar: train_loader y test_loader
# -------------------------------------------------------------------
seed = int(time.time_ns() % (2**32))
torch.manual_seed(seed)
np.random.seed(seed)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\n✅ Datos VAE cargados con éxito:")
print(f"Dimensión de características numéricas: {input_dim}")
print(f"Número de clases: {num_classes}")
print("Distribución Entrenamiento:", Counter(y_train))
print("Distribución Evaluación:", Counter(y_test))

Usando dispositivo: cuda

✅ Datos VAE cargados con éxito:
Dimensión de características numéricas: 768
Número de clases: 7
Distribución Entrenamiento: Counter({np.int8(0): 400, np.int8(2): 291, np.int8(5): 160, np.int8(4): 118, np.int8(3): 75, np.int8(6): 30, np.int8(1): 18})
Distribución Evaluación: Counter({np.int8(0): 100, np.int8(2): 54, np.int8(5): 30, np.int8(4): 22, np.int8(3): 14, np.int8(6): 3, np.int8(1): 1})


In [10]:
# Celda 3: LightGBM
print("🚀 Iniciando Grid Search para LIGHTGBM...")

param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 9, -1],
    'num_leaves': [15, 31, 63],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgbm = GridSearchCV(
    estimator=LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1),
    param_grid=param_grid_lgbm,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_lgbm.fit(X_train_vec, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (LightGBM):")
print(grid_lgbm.best_params_)

best_lgbm = grid_lgbm.best_estimator_
y_pred_lgbm = best_lgbm.predict(X_test_vec)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (LightGBM):")
print(classification_report(y_test, y_pred_lgbm, target_names=codes))

🚀 Iniciando Grid Search para LIGHTGBM...
Fitting 5 folds for each of 432 candidates, totalling 2160 fits


ValueError: 
All the 2160 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2160 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
    ~~~~~~~~~~~^
        X,
        ^^
    ...<12 lines>...
        init_model=init_model,
        ^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\lightgbm\sklearn.py", line 949, in fit
    _X, _y = _LGBMValidateData(
             ~~~~~~~

In [14]:
# Celda 4: XGBoost
print("🚀 Iniciando Grid Search para XGBOOST...")

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=seed, eval_metric='mlogloss', n_jobs=-1),
    param_grid=param_grid_xgb,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_xgb.fit(X_train_vec, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (XGBoost):")
print(grid_xgb.best_params_)

best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test_vec)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (XGBoost):")
print(classification_report(y_test, y_pred_xgb, target_names=codes))

🚀 Iniciando Grid Search para XGBOOST...
Fitting 5 folds for each of 729 candidates, totalling 3645 fits


ValueError: 
All the 3645 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
3645 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\core.py", line 729, in inner_f
    return func(**kwargs)
  File "c:\Users\Usuario\miniforge3\envs\alba\Lib\site-packages\xgboost\sklearn.py", line 1664, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
                           ~~~~~~~~~~~~~~~~~~~~~~~~~^
        missing=self.missing,
        ^^^^^^^^^^^^^^^^^^^^^
    ...<14

In [ ]:
# Celda 5: Gradient Boosting
print("🚀 Iniciando Grid Search para GRADIENT BOOSTING (Scikit-Learn)...")

param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 1.0]
}

grid_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=seed),
    param_grid=param_grid_gb,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_gb.fit(X_train_vec, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (GradientBoosting):")
print(grid_gb.best_params_)

best_gb = grid_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test_vec)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (GradientBoosting):")
print(classification_report(y_test, y_pred_gb, target_names=codes))

🚀 Iniciando Grid Search para GRADIENT BOOSTING (Scikit-Learn)...
Fitting 5 folds for each of 144 candidates, totalling 720 fits

⏱️ Búsqueda completada en 224.27 segundos.
🏆 MEJORES HIPERPARÁMETROS (GradientBoosting):
{'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.8}

📊 REPORTE DE CLASIFICACIÓN EN TEST (GradientBoosting):
              precision    recall  f1-score   support

         F20       0.56      0.77      0.65       150
         F21       0.00      0.00      0.00         2
         F22       0.39      0.32      0.35        81
         F23       0.12      0.10      0.11        21
         F25       0.45      0.27      0.34        33
         F29       0.33      0.16      0.21        45
       F60.1       0.00      0.00      0.00         4

    accuracy                           0.48       336
   macro avg       0.26      0.23      0.24       336
weighted avg       0.44      0.48      0.44       336

